# 1D L2 Inversion with real data 

Workflow to do L2 inversion in real line data

In [3]:
# SimPEG functionality
import simpeg.electromagnetics.frequency_domain as fdem
from simpeg.utils import plot_1d_layer_model, download, mkvc
from simpeg import (
    maps,
    data,
    data_misfit,
    regularization,
    optimization,
    inverse_problem,
    inversion,
    directives,
)

# discretize functionality
from discretize import TensorMesh

# Common Python functionality
import os
import numpy as np
import pandas as pd
from scipy.constants import mu_0
import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import ipywidgets
import dill
%matplotlib inline
mpl.rcParams.update({"font.size": 12})

#Import helper functions
import sys
import data_functions, inversion_functions


# Read the data:

Read from csv file of block 53:

In [5]:
# Get the current working directory (where the notebook is running)
current_dir = os.getcwd()

# Move one level up to the parent directory
parent_dir = os.path.dirname(current_dir)

# Construct the full path to the CSV file in the parent directory
csv_path = os.path.join(parent_dir, "block53_fdem_inv.csv")

fdem_data = pd.read_csv(csv_path) ## Set to the position of the csv_files list where the database you want is
pd.set_option('display.max_columns', len(fdem_data.columns))
fdem_data.head()

/var/folders/gb/v44v1_q914718nzwmm90d6pr0000gn/T/ipykernel_23784/3595362424.py:10: DtypeWarning: Columns (80,86,87,311,312) have mixed types. Specify dtype option on import or set low_memory=False.
  fdem_data = pd.read_csv(csv_path) ## Set to the position of the csv_files list where the database you want is


,x_tx,y_tx,gpsz_tx,altlas_bird,altlas_bird_derot,altrad_heli,bird_pitch,bird_roll,bird_yaw,Block,cp40k_reim,cp140k_reim,cp400_reim,cp1800_reim,cp8200_reim,cpi40k,cpi140k,cpi400,cpi1800,cpi8200,cpq40k,cpq140k,cpq400,cpq1800,cpq8200,cpsp,cx3300_reim,cxi3300,cxq3300,cxsp,date,ddep40k,ddep140k,ddep400,ddep1800,ddep8200,dep40k,dep140k,dep400,dep1800,dep8200,diurnal,diurnal_cor,dtm,fid,flight,gpsz_heli,igrf,lat_heli,lat_tx,Line,line(1),long_heli,long_tx,mag_diu,mag_lag,mag_rmi,mag_rmi analytic signal,mag_rmi hd filtered,mag_rmi rtp filtered,mag_rmi tilt angle,mag_rmi vd filtered,mag_rmi_HDX,mag_rmi_HDY,mag_tmi,powerline,res40k,res140k,res400,res1800,res8200,time_ssm,x_heli,y_heli,iter,beta,phi_d,phi_m,f,n_layers,resistivies,resistivity_hsp,phi_d_star,chi_factor,rms,doi_CA2012,Sj_star_CA2012,S_CA2012,rho01,rho02,rho03,rho04,rho05,rho06,rho07,rho08,rho09,rho10,rho11,rho12,rho13,rho14,rho15,rho16,rho17,rho18,rho19,rho20,rho21,rho22,rho23,rho24,rho25,rho26,rho27,rho28,rho29,rho30,rho31,rho32,rho33,rho34,rho35,rho36,rho37,rho38,rho39,rho40,thick01,thick02,thick03,thick04,thick05,thick06,thick07,thick08,thick09,thick10,thick11,thick12,thick13,thick14,thick15,thick16,thick17,thick18,thick19,thick20,thick21,thick22,thick23,thick24,thick25,thick26,thick27,thick28,thick29,thick30,thick31,thick32,thick33,thick34,thick35,thick36,thick37,thick38,thick39,dpred_cpi140k,dpred_cpq140k,dpred_cpi40k,dpred_cpq40k,dpred_cpi8200,dpred_cpq8200,dpred_cpi1800,dpred_cpq1800,dpred_cpi400,dpred_cpq400,dobs_cpi140k,dobs_cpq140k,dobs_cpi40k,dobs_cpq40k,dobs_cpi8200,dobs_cpq8200,dobs_cpi1800,dobs_cpq1800,dobs_cpi400,dobs_cpq400,rho_halfspace,rho01_mref1,rho02_mref1,rho03_mref1,rho04_mref1,rho05_mref1,rho06_mref1,rho07_mref1,rho08_mref1,rho09_mref1,rho10_mref1,rho11_mref1,rho12_mref1,rho13_mref1,rho14_mref1,rho15_mref1,rho16_mref1,rho17_mref1,rho18_mref1,rho19_mref1,rho20_mref1,rho21_mref1,rho22_mref1,rho23_mref1,rho24_mref1,rho25_mref1,rho26_mref1,rho27_mref1,rho28_mref1,rho29_mref1,rho30_mref1,rho31_mref1,rho32_mref1,rho33_mref1,rho34_mref1,rho35_mref1,rho36_mref1,rho37_mref1,rho38_mref1,rho39_mref1,rho40_mref1,rho_hsp_mref1,rho01_mref2,rho02_mref2,rho03_mref2,rho04_mref2,rho05_mref2,rho06_mref2,rho07_mref2,rho08_mref2,rho09_mref2,rho10_mref2,rho11_mref2,rho12_mref2,rho13_mref2,rho14_mref2,rho15_mref2,rho16_mref2,rho17_mref2,rho18_mref2,rho19_mref2,rho20_mref2,rho21_mref2,rho22_mref2,rho23_mref2,rho24_mref2,rho25_mref2,rho26_mref2,rho27_mref2,rho28_mref2,rho29_mref2,rho30_mref2,rho31_mref2,rho32_mref2,rho33_mref2,rho34_mref2,rho35_mref2,rho36_mref2,rho37_mref2,rho38_mref2,rho39_mref2,rho40_mref2,rho_hsp_mref2,doi_index_layer01,doi_index_layer02,doi_index_layer03,doi_index_layer04,doi_index_layer05,doi_index_layer06,doi_index_layer07,doi_index_layer08,doi_index_layer09,doi_index_layer10,doi_index_layer11,doi_index_layer12,doi_index_layer13,doi_index_layer14,doi_index_layer15,doi_index_layer16,doi_index_layer17,doi_index_layer18,doi_index_layer19,doi_index_layer20,doi_index_layer21,doi_index_layer22,doi_index_layer23,doi_index_layer24,doi_index_layer25,doi_index_layer26,doi_index_layer27,doi_index_layer28,doi_index_layer29,doi_index_layer30,doi_index_layer31,doi_index_layer32,doi_index_layer33,doi_index_layer34,doi_index_layer35,doi_index_layer36,doi_index_layer37,doi_index_layer38,doi_index_layer39,doi_index_layer40,dist,depths,alts,depth01,alt01,depth02,alt02,depth03,alt03,depth04,alt04,depth05,alt05,depth06,alt06,depth07,alt07,depth08,alt08,depth09,alt09,depth10,alt10,depth11,alt11,depth12,alt12,depth13,alt13,depth14,alt14,depth15,alt15,depth16,alt16,depth17,alt17,depth18,alt18,depth19,alt19,depth20,alt20,depth21,alt21,depth22,alt22,depth23,alt23,depth24,alt24,depth25,alt25,depth26,alt26,depth27,alt27,depth28,alt28,depth29,alt29,depth30,alt30,depth31,alt31,depth32,alt32,depth33,alt33,depth34,alt34,depth35,alt35,depth36,alt36,depth37,alt37,depth38,alt38,depth39,alt39,depth40,alt40,depth41,alt41,doi_OL1999,doi_alt_OL1999,doi_alt_CA2012,x,y,z,layer,rho
0,553008.70

In [6]:
#Rename columns if needed:
#fdem_data.rename(columns={'X': 'x_tx', 'Y':'y_tx', 'Z':'gpsz_tx'}, inplace=True)

#Show all columns:
#np.array(fdem_data.columns)

In [29]:
# Set Halfspace conductivity:
conductivity_hsp = 0.024795715924493058
resistivity_hsp = 1/conductivity_hsp

# Inversion Process

Run inversions in couple lines to define best beta values to run line inversion. 

In [21]:
# Define like working with:
line_no = 'L530100'
line = fdem_data.loc[fdem_data['Line'] == line_no].copy()

# Calculate the tx_rx coordinates:
line = data_functions.calculate_tx_rx_coordinates(line)

#Filter out nan values:
line = line[line['fid'].notna()]

# Get fid values
line_stations = line.fid.values

#Choose every 400 lines
line_betas = line_stations[::400]

print(line_betas)

# See last beta value: inv_prob.beta
#Set beta in inv_prob as other parameter and get rid of beta cooling (BetaEstimate_ByEig and BetaSchedule)

[2319. 2359. 2399. 2439. 2479. 2347. 2387. 2427. 2467.]


In [27]:
# Loop through the line_betas with the inversion process
for fid_n in line_betas:

    #choose line
    line_station = line.loc[line.fid == fid_n].iloc[0]

    #Calculate the height (terrain clearance)
    line_station.loc['height_tx'] = line_station['gpsz_tx'] - line_station['dtm']

    ### Define Survey ###
    #Define the surveys and freq to avoid (none codes)
    freq_avoid = []
    survey = inversion_functions.survey_object(line_station, freq_avoid)

    ### Define Data Object ###
    # Defines the data object and assigns uncertainties with the noise floor
    uncertainty_floor = 5.0e0
    relative_error = 0.05 #What is relative error?

    data_object = inversion_functions.data_object(line_station, relative_error = relative_error, noise_floor = uncertainty_floor, survey = survey, freq_avoid = freq_avoid)

    ### Mesh Definition for L2 Inversion ###

    #Calculate skin depth:
    ### Depth max ###
    skin_depth = 503*np.sqrt(resistivities_hsp/np.min(frequencies))
    depth_max = 2*skin_depth  # depth to lowest layer
    



## Defining Data Object


In [ ]:
# Defines the data object and assigns uncertainties with the noise floor
uncertainty_floor = 5.0e0
relative_error = 0.05 #What is relative error?

data_object = inversion_functions.data_object(line_station, relative_error = relative_error, noise_floor = uncertainty_floor, survey = survey, freq_avoid = freq_avoid)

# Halfspace Inversion

In [16]:
### Mesh Definition for halfspace ###
layer_thick_halfspace = [1000]
n_layers_halfspace = 1

regularization_mesh_hs, log_conductivity_halfspace_map, simulation_hsp_L2 = inversion_functions.mapping_forward_objects(layer_thicknesses = layer_thick_halfspace, n_layers = n_layers_halfspace, survey = survey)


In [17]:
# Starting model is log-conductivity values (S/m)
starting_conductivity_model_hsp = np.log(1e-3 * np.ones(n_layers_halfspace))

# Reference model, same as starting 
reference_conductivity_model_hsp = starting_conductivity_model_hsp.copy()

## Data Misfit 

In [19]:
dmis_hsp_L2 = data_misfit.L2DataMisfit(simulation=simulation_hsp_L2, data=data_object)

## Regularization

In [21]:
reg_L2 = inversion_functions.regularization_object(regularization_mesh_hs, reference_conductivity_model = reference_conductivity_model_hsp , alpha_s = 1e-5, alpha_x = 1)

## Optimization

In [23]:
opt_L2 = optimization.InexactGaussNewton(
    maxIter=100, maxIterLS=20, maxIterCG=20, tolCG=1e-3
)

## Inversion Parameters

In [25]:
# Combine the inverse problem and the set of directives
inv_L2 = inversion_functions.inversion_setup(dmis_hsp_L2, reg_L2, opt_L2)

## Running Inversion

In [82]:
%%capture 
#To not show output

recovered_halfspace_model_L2 = inv_L2.run(starting_conductivity_model_hsp)

## Get the recovered halfspace resistivity from model estimated
conductivities_hsp = log_conductivity_halfspace_map * recovered_halfspace_model_L2
resistivities_hsp = 1 / conductivities_hsp

## Plot results

In [78]:
#line_no = "fake #"
#fid_n = "fake sounding"
print(f"\n **** Halfspace inversion for line {line_no} and sounding {fid_n}... ****")
print("Resistivity halfspace: ", resistivities_hsp)



 **** Halfspace inversion for line L530100 and sounding 2490.9... ****
Resistivity halfspace:  [32.21509256]


In [ ]:
#> /dev/null

# ToDos
* Use paralelizations and GPUs